# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [1]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets.dataset_dict import DatasetDict
from datasets.arrow_dataset import Dataset
from datasets import load_dataset
import pandas as pd

import datasets
import os

## Baseline models

In [2]:
from transformers import AutoTokenizer, AutoModel

In [3]:
def chunk_text(text, tokeizer, chunk_size=512, overlap=62):

    token_ids = tokeizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + chunk_size
        chunk_tokens = token_ids[start:end]
        chunk_text = tokeizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start += chunk_size - overlap

    return chunks

In [4]:
class XMLRoBERTa():

    name = "xlm-roberta-large"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large")
        self.tokenizer = AutoTokenizer.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float32)
        self.model.eval()

    def encode(
            self,
            text,
            strategy,
            **kwargs) -> torch.tensor:

        if strategy == "first":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")
            # select first chunk_size tokens
            tokenized_first_input_ids = tokenized_text["input_ids"][:, :self.chunk_size]
            attention_mask = tokenized_text["attention_mask"][:, :self.chunk_size]
            with torch.no_grad():
                outputs = self.model(tokenized_first_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "last":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")
            # select last chunk_size tokens
            tokenized_last_input_ids = tokenized_text["input_ids"][:, -self.chunk_size:]
            attention_mask = tokenized_text["attention_mask"][:, -self.chunk_size:]
            with torch.no_grad():
                outputs = self.model(tokenized_last_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "chunking":
            # for text in input make a chunking
            texts_chunked = chunk_text(text, self.tokenizer, self.chunk_size)

            # perform a tokeniztaion of each chunk
            chunk_input_ids = []
            chunk_attention_masks = []

            tokenized = self.tokenizer(
                texts_chunked,    # texts_chunked is effectively a batch of chunks at this point
                padding=True,
                truncation=True,
                max_length=self.chunk_size,
                return_tensors="pt"
            )

            # get embeddings for all chunks
            with torch.no_grad():
                outputs = self.model(**tokenized)
                # mean pooling for each chunk
                embedding = outputs.last_hidden_state.mean(dim=1)
                # mean pooling across chunks:
                embedding = embedding.mean(dim=0)
                return embedding


class Qwen3_Embedding():

    name = "Qwen3-Embedding-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size

        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float32)

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states, attention_mask):
        return last_hidden_states[:, -1]

    def encode(
            self,
            text,
            strategy = "chunking",
            **kwargs) -> torch.tensor:

        if strategy == "first":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")
            # select first chunk_size tokens
            tokenized_first_input_ids = tokenized_text["input_ids"][:, :self.chunk_size]
            attention_mask = tokenized_text["attention_mask"][:, :self.chunk_size]

            with torch.no_grad():
                outputs = self.model(tokenized_first_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "last":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")

            tokenized_first_input_ids = tokenized_text["input_ids"][:, -self.chunk_size:]
            attention_mask = tokenized_text["attention_mask"][:, -self.chunk_size:]

            with torch.no_grad():
                outputs = self.model(tokenized_first_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "chunking":
            text_chunked = chunk_text(text, self.tokenizer, self.chunk_size)

            # perform a tokeniztaion of each chunk
            chunk_input_ids = []
            chunk_attention_masks = []

            tokenized = self.tokenizer(
                text_chunked,    # texts_chunked is effectively a batch of chunks at this point
                padding=True,
                truncation=True,
                max_length=self.chunk_size,
                return_tensors="pt"
            )

            # get embeddings for all chunks
            with torch.no_grad():
                outputs = self.model(**tokenized)
                # get an embegging from end of sequence token
                embedding = self.__get_eos_token_embedding(outputs.last_hidden_state, tokenized["attention_mask"])
                # mean pooling for each chunk
                embedding = embedding.mean(dim=0)
                return embedding

## LongEmbed LEMBWikimQARetrieval

In [5]:
from datasets import load_dataset

ds = load_dataset("dwzhu/LongEmbed", name="2wikimqa")
corpus = ds["corpus"]
queries = ds["queries"]
qrels = ds["qrels"]

In [6]:
qwen3_embed = Qwen3_Embedding(512)
xlm_roberta = XMLRoBERTa(512)

### Encoding document with each model with each text-preprocess strategy

In [7]:
# encode each document in a corpus
def encode_documents(corpus, model, strategy):
    document_embeddings = {}
    for doc in corpus:
        embedding = model.encode(doc["text"], strategy)
        document_embeddings[doc["doc_id"]] = embedding.numpy()
    return document_embeddings

# encode each query
def encode_queries(queries, model):
    queries_embeddings = {}
    for query in queries:
        embedding = model.encode(query["text"], strategy="chunking")
        queries_embeddings[query["qid"]] = embedding.numpy()
    return queries_embeddings

In [8]:
# encoding the queries with each mode
# strategy does not really matter in this case since the query will be one chunk long anyway
q3_query_embed = encode_queries(queries, qwen3_embed)
roberta_query_embed = encode_queries(queries, xlm_roberta)

In [15]:
pd.DataFrame(q3_query_embed).to_csv("./query_embed/q3_query_embed.csv")
pd.DataFrame(roberta_query_embed).to_csv("./query_embed/roberta_query_embed.csv")

In [ ]:
q3_doc_embed_chunking = encode_documents(corpus, qwen3_embed, strategy="chunking")
roberta_doc_embed_chunking = encode_documents(corpus, xlm_roberta, strategy="chunking")

In [ ]:
pd.DataFrame(q3_doc_embed_chunking).to_csv("q3_doc_embed_chunking.csv")
pd.DataFrame(roberta_doc_embed_chunking).to_csv("roberta_doc_embed_chunking.csv")

In [ ]:
q3_doc_embed_first = encode_documents(corpus, qwen3_embed, strategy="first")
roberta_doc_embed_first = encode_documents(corpus, xlm_roberta, strategy="first")

In [ ]:
pd.DataFrame(q3_doc_embed_first).to_csv("q3_doc_embed_first.csv")
pd.DataFrame(roberta_doc_embed_first).to_csv("roberta_doc_embed_first.csv")

In [ ]:
q3_doc_embed_last = encode_documents(corpus, qwen3_embed, strategy="last")
roberta_doc_embed_last = encode_documents(corpus, xlm_roberta, strategy="last")

In [ ]:
pd.DataFrame(q3_doc_embed_last).to_csv("q3_doc_embed_last.csv")
pd.DataFrame(roberta_doc_embed_last).to_csv("roberta_doc_embed_last.csv")

### Calculating evaluation metrics

In [ ]:
from sklear.metrics.pairwise import cosine_similarity

In [16]:
def MAP_at_K(documents_embed, queries_embed, qrel, k):
    sum_ap_at_k = 0
    n_queries = 0
    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = queries_embed[query_id]
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = documents_embed[doc_id]
            cos_sim = cosine_similarity(q_emebedding, doc_embedding)
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
        except ValueError:
            true_document_position = -1
        ap_at_k = 1 / true_document_position
        sum_ap_at_k += ap_at_k
        n_queries += 1
    map = sum_ap_at_k / n_queries
    return map

fatal: not a git repository (or any of the parent directories): .git


In [ ]:
def mean_nDCG_at_k(documents_embed, queries_embed, qrel, k):
    sum_ndcg_at_k = 0
    n_queries = 0

    idcg_at_k = 0
    for i in range(k):
        idcg_at_k += 1 / np.log2(i + 2)

    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = queries_embed[query_id]
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = documents
            cos_sim = cosine_similarity(q_emebedding, doc_embedding)
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
        except ValueError:
            true_document_position = -1
        dcg_at_k = 1 / np.log2(true_document_position + 1)
        ndcg_at_k = dcg_at_k / idcg_at_k
        sum_ndcg_at_k += ndcg_at_k
        n_queries += 1

    mean_ndcg = sum_ndcg_at_k / n_queries
    return mean_ndcg